# ITM 454 - Natural Language Processing
## Named Entity Recognition (NER) with Conditional Random Field (CRF)

---

Reference: https://eli5.readthedocs.io/en/latest/tutorials/sklearn_crfsuite.html

In [2]:
%pip install sklearn_crfsuite

Note: you may need to restart the kernel to use updated packages.


In [3]:
import nltk
import sklearn_crfsuite

### 1. Training Data
Since CoNLL2003 is not directly available in nltk due to licensing. So, we have to download and load it manually.

In [4]:
def read_conll_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        sent_list = []
        sent = []
        for line in f:
            line = line.strip()
            if "DOCSTART" in line: continue
            if line != "":
                word, pos, _, ner = line.split(" ")
                sent.append((word, pos, ner))
            else:
                if sent:
                    sent_list.append(sent)
                sent = []

    return sent_list

train_sents = read_conll_file("/Users/soryapheakly/Downloads/CONLL-2003/train.txt")
test_sents = read_conll_file("/Users/soryapheakly/Downloads/CONLL-2003/test.txt")
print(train_sents[0])
print(test_sents[0])

[('EU', 'NNP', 'B-ORG'), ('rejects', 'VBZ', 'O'), ('German', 'JJ', 'B-MISC'), ('call', 'NN', 'O'), ('to', 'TO', 'O'), ('boycott', 'VB', 'O'), ('British', 'JJ', 'B-MISC'), ('lamb', 'NN', 'O'), ('.', '.', 'O')]
[('SOCCER', 'NN', 'O'), ('-', ':', 'O'), ('JAPAN', 'NNP', 'B-LOC'), ('GET', 'VB', 'O'), ('LUCKY', 'NNP', 'O'), ('WIN', 'NNP', 'O'), (',', ',', 'O'), ('CHINA', 'NNP', 'B-PER'), ('IN', 'IN', 'O'), ('SURPRISE', 'DT', 'O'), ('DEFEAT', 'NN', 'O'), ('.', '.', 'O')]


### 2. Feature Extraction

In [5]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,                       # constant feature, lets the CRF learn a per-label baseline weight
        'word.lower()': word.lower(),      # lowercased word, so "Apple" and "apple" share this feature
        'word[-3:]': word[-3:],            # last 3 characters, a cheap suffix/morphology cue (e.g. "-ing", "-ton")
        'word.isupper()': word.isupper(),  # True if the whole word is uppercase (e.g. acronyms like "EU")
        'word.istitle()': word.istitle(),  # True if the word is capitalized like a name (e.g. "German")
        'word.isdigit()': word.isdigit(),  # True if the word is a number
        'postag': postag,                  # full part-of-speech tag (e.g. "NNP")
        'postag[:2]': postag[:2],          # coarse POS tag, first 2 chars (e.g. "NN" covers NN, NNP, NNS)
    }
    if i > 0:
        # features describing the previous word, giving the CRF left-context
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True  # marks this as the Beginning Of Sentence (no previous word)

    if i < len(sent)-1:
        # features describing the next word, giving the CRF right-context
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True  # marks this as the End Of Sentence (no next word)

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    return [token for token, postag, label in sent]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

In [6]:
X_train[1]

[{'bias': 1.0,
  'word.lower()': 'peter',
  'word[-3:]': 'ter',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'NNP',
  'postag[:2]': 'NN',
  'BOS': True,
  '+1:word.lower()': 'blackburn',
  '+1:word.istitle()': True,
  '+1:word.isupper()': False,
  '+1:postag': 'NNP',
  '+1:postag[:2]': 'NN'},
 {'bias': 1.0,
  'word.lower()': 'blackburn',
  'word[-3:]': 'urn',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'NNP',
  'postag[:2]': 'NN',
  '-1:word.lower()': 'peter',
  '-1:word.istitle()': True,
  '-1:word.isupper()': False,
  '-1:postag': 'NNP',
  '-1:postag[:2]': 'NN',
  'EOS': True}]

### 3. Training the CRF Model

In [7]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=20,
    all_possible_transitions=False,
)
crf.fit(X_train, y_train)

,algorithm,'lbfgs'
,all_possible_transitions,False
,c1,0.1
,c2,0.1
,max_iterations,20
,min_freq,None
,all_possible_states,None
,num_memories,None
,epsilon,None
,period,None
,delta,None


### 4. Evaluation

In [8]:
from sklearn_crfsuite import metrics

# Make predictions on test set
y_pred_test = crf.predict(X_test)

print("Test Set Performance:")
print("=" * 40)

# Token-level accuracy
test_accuracy = metrics.flat_accuracy_score(y_test, y_pred_test)
print(f"Token-level accuracy: {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")

# Entity-level metrics
test_f1 = metrics.flat_f1_score(y_test, y_pred_test, average='weighted')
print(f"Weighted F1-score: {test_f1:.3f}")

print("\nDetailed Test Classification Report:")
print(metrics.flat_classification_report(y_test, y_pred_test, digits=3))

Test Set Performance:
Token-level accuracy: 0.921 (92.1%)
Weighted F1-score: 0.922

Detailed Test Classification Report:
              precision    recall  f1-score   support

       B-LOC      0.637     0.704     0.669      1668
      B-MISC      0.735     0.594     0.657       702
       B-ORG      0.605     0.568     0.586      1661
       B-PER      0.750     0.696     0.722      1617
       I-LOC      0.558     0.467     0.508       257
      I-MISC      0.429     0.421     0.425       216
       I-ORG      0.427     0.668     0.521       835
       I-PER      0.817     0.784     0.800      1156
           O      0.981     0.976     0.979     38323

    accuracy                          0.921     46435
   macro avg      0.660     0.653     0.652     46435
weighted avg      0.925     0.921     0.922     46435



### 5. Examples from the Test Set

Numbers are nice, but it's easier to understand NER by just looking at a sentence with its predicted tags.
Below we pick a few sentences from the test set and print each word next to the **true** label and the **predicted** label, so you can see where the model gets it right (or wrong).

In [9]:
def show_sentence(sent, true_labels, pred_labels):
    words = sent2tokens(sent)
    print(f"{'WORD':<15}{'TRUE':<10}{'PREDICTED':<10}")
    print("-" * 35)
    for word, true_tag, pred_tag in zip(words, true_labels, pred_labels):
        mark = "" if true_tag == pred_tag else "  <-- different"
        print(f"{word:<15}{true_tag:<10}{pred_tag:<10}{mark}")
    print()

# Show the first 3 sentences from the test set
for i in range(3):
    print(f"Example {i+1}: {' '.join(sent2tokens(test_sents[i]))}")
    print()
    show_sentence(test_sents[i], y_test[i], y_pred_test[i])

Example 1: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .

WORD           TRUE      PREDICTED 
-----------------------------------
SOCCER         O         O         
-              O         O         
JAPAN          B-LOC     B-ORG       <-- different
GET            O         O         
LUCKY          O         B-LOC       <-- different
WIN            O         I-LOC       <-- different
,              O         O         
CHINA          B-PER     B-LOC       <-- different
IN             O         O         
SURPRISE       O         O         
DEFEAT         O         O         
.              O         O         

Example 2: Nadim Ladki

WORD           TRUE      PREDICTED 
-----------------------------------
Nadim          B-PER     B-PER     
Ladki          I-PER     I-PER     

Example 3: AL-AIN , United Arab Emirates 1996-12-06

WORD           TRUE      PREDICTED 
-----------------------------------
AL-AIN         B-LOC     B-LOC     
,              O         O         

### 6. Try Your Own Sentence!

Write any sentence below and see what the trained model predicts.

In [10]:
def predict_ner(sentence):
    words = nltk.word_tokenize(sentence)
    tagged = nltk.pos_tag(words)
    sent = [(word, postag, "O") for word, postag in tagged]

    pred_labels = crf.predict([sent2features(sent)])[0]

    print(f"{'WORD':<15}{'PREDICTED':<10}")
    print("-" * 25)
    for word, tag in zip(words, pred_labels):
        print(f"{word:<15}{tag:<10}")


# Change this sentence to whatever you like!
my_sentence = "Elon Musk visited Paris last summer to meet officials from Google."
predict_ner(my_sentence)

WORD           PREDICTED 
-------------------------
Elon           B-PER     
Musk           I-PER     
visited        O         
Paris          B-PER     
last           O         
summer         O         
to             O         
meet           O         
officials      O         
from           O         
Google         B-LOC     
.              O         


**Features description comments, section 5 & 6 are generated by Claude Sonnet 5.